# 🔍 Watermark Detection Analysis

## 📦 Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from pathlib import Path
import ast
import os
import sys
import importlib.util
import re
import math
from collections import Counter

## ⚙️ Configuration

**Update these parameters based on your experiment:**

In [ ]:
# === EXPERIMENT CONFIGURATION ===

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = '/home/fahad/Documents/MASTERS/CODEBLOCKS/Masters-Thesis-Code/promptMark'

#! Experiment parameters
MODEL_NAME = "gemini"
EXPERIMENT_NUMBER = 'exp1-1'
EXP_VERSION = 'v1'
GENERATION_TYPE = 'during'  # 'during' or 'after'
SAMPLE_SIZE = 100
DATASET = 'mbpp'
PROCESSED_SIZE = 5  # Actual number of samples processed (subset of dataset)

GREEN_WORDS = ['billions', 'dlrs', 'shade', 'trade', 'profit']
RED_WORDS = ['market', 'year', 'company', 'revs']

# Watermark parameters
Z_THRESHOLD = 2.120  # Adjust based on your calibration done in `calculate_auroc_v2.py`
GAMMA = 0.5  # From NLTK Brown corpus

# Derived paths
DATASET_FILE = f'sanitized-mbpp-sample-100.json'
DATASET_PATH = f'{BASE_DIR}/datasets/core/{DATASET_FILE}'
OUTPUT_DIR = f'{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}'
RESULTS_CSV = f'{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv'

## 🛠️ Core Functions

In [ ]:
def calculate_z_score(token_count, green_count, gamma=GAMMA):
    """Calculate z-score for watermark detection"""
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(gamma * (1 - gamma) * token_count)

def extract_words_from_text(code_text):
    """Extract all words from code treating it as plain text"""
    # Extract all word-like tokens (alphanumeric sequences)
    words = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', code_text)
    # Filter out very short words and common single characters
    filtered_words = [word for word in words if len(word) > 1]
    return filtered_words

def analyze_code_watermark(code, verbose=False):
    """Analyze watermark in a single code sample"""
    words = extract_words_from_text(code)
    if not words:
        return {
            "z_score": float("nan"),
            "is_watermarked": False,
            "word_count": 0,
            "green_count": 0,
            "starting_letters": []
        }
    
    # Get starting letters of all words
    starting_letters = [word[0].lower() for word in words if word and word[0].isalpha()]
    
    # Count green letters (based on GREEN_WORDS first letters)
    green_letters = set([word[0].lower() for word in GREEN_WORDS])
    green_count = sum(1 for letter in starting_letters if letter in green_letters)
    
    z_score = calculate_z_score(len(starting_letters), green_count)
    is_watermarked = z_score >= Z_THRESHOLD if not math.isnan(z_score) else False
    
    if verbose:
        print(f"Words found: {len(words)}")
        print(f"Starting letters: {len(starting_letters)}")
        print(f"Green letters count: {green_count}")
        print(f"Z-score: {z_score:.3f}")
        print(f"Is watermarked: {is_watermarked}")
    
    return {
        "z_score": z_score,
        "is_watermarked": is_watermarked,
        "word_count": len(words),
        "green_count": green_count,
        "starting_letters": starting_letters
    }

def run_code_tests(code, test_cases):
    """Run test cases on generated code"""
    results = []
    temp_file = 'temp_test.py'
    
    with open(temp_file, 'w', encoding='utf-8') as f:
        f.write(code + '\n')
    
    try:
        spec = importlib.util.spec_from_file_location("temp_test", temp_file)
        module = importlib.util.module_from_spec(spec)
        sys.modules["temp_test"] = module
        spec.loader.exec_module(module)
        
        globals().update(vars(module))
        
        for idx, test in enumerate(test_cases, 1):
            try:
                exec(test, globals())
                results.append((idx, True, None))
            except Exception as e:
                results.append((idx, False, str(e)))
    
    except Exception as e:
        for idx in range(len(test_cases)):
            results.append((idx+1, False, f"Code error: {str(e)}"))
    
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)
    
    return results

print("✅ Core functions loaded successfully")

## 📊 Data Loading and Preparation

In [ ]:
# Load dataset
print("📂 Loading dataset...")
df = pd.read_json(DATASET_PATH, lines=True)
print(f"Dataset loaded: {len(df)} samples")

# Check if results CSV exists
if os.path.exists(RESULTS_CSV):
    print(f"📂 Loading existing results from {RESULTS_CSV}")
    results_df = pd.read_csv(RESULTS_CSV)
    print(f"Results loaded: {len(results_df)} samples")
else:
    print(f"⚠️ Results CSV not found at {RESULTS_CSV}")
    print("Will analyze generated code files directly...")
    results_df = None

# Verify output directory
output_path = Path(OUTPUT_DIR)
if output_path.exists():
    generated_files = list(output_path.glob("*.py"))
    print(f"📂 Found {len(generated_files)} generated code files")
else:
    print(f"❌ Output directory not found: {OUTPUT_DIR}")
    generated_files = []

## 🔄 Analysis: Generated Code

If results CSV doesn't exist, analyze generated code files directly:

In [ ]:
if results_df is None and generated_files:
    print("🔄 Analyzing generated code files...")
    analysis_results = []
    
    for _, row in df.iterrows():
        task_id = row.get('task_id') or row.get('id') or str(_)
        original_code = row.get('code', '')
        
        # Find corresponding generated file
        generated_file = output_path / f"{task_id}.py"
        
        if not generated_file.exists():
            print(f"⚠️ Missing generated file for task {task_id}")
            continue
            
        # Load generated code
        generated_code = generated_file.read_text(encoding='utf-8')
        
        # Analyze watermarks
        orig_analysis = analyze_code_watermark(original_code)
        gen_analysis = analyze_code_watermark(generated_code)
        
        # Run tests if available
        test_results = run_code_tests(generated_code, row.get('test_list', []))
        passed = sum(1 for _, success, _ in test_results if success)
        total = len(test_results)
        
        result = {
            'task_id': task_id,
            'original_z_score': orig_analysis['z_score'],
            'generated_z_score': gen_analysis['z_score'],
            'original_is_watermarked': orig_analysis['is_watermarked'],
            'generated_is_watermarked': gen_analysis['is_watermarked'],
            'tests_passed': passed,
            'tests_failed': total - passed,
            'total_tests': total,
            'pass_rate': round((passed / total * 100) if total > 0 else 0, 2),
            'all_passed': (passed == total),
            'original_word_count': orig_analysis['word_count'],
            'generated_word_count': gen_analysis['word_count'],
            'original_green_count': orig_analysis['green_count'],
            'generated_green_count': gen_analysis['green_count']
        }
        analysis_results.append(result)
    
    # Create results DataFrame
    results_df = pd.DataFrame(analysis_results)
    
    # Save results
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"💾 Results saved to {RESULTS_CSV}")

print(f"📊 Analysis complete: {len(results_df) if results_df is not None else 0} samples processed")

## 📈 Performance Analysis

### Confusion Matrix Calculation

In [ ]:
# results_df.columns

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_fscore_support

def calculate_confusion_metrics(original_z_scores, generated_z_scores, z_threshold):
    # Combine data
    all_z_scores = np.concatenate([original_z_scores, generated_z_scores])
    true_labels = np.concatenate([
        np.zeros(len(original_z_scores)),  # Original = 0 (not watermarked)
        np.ones(len(generated_z_scores))   # Generated = 1 (watermarked)
    ])
    
    # Predictions based on threshold
    predictions = (all_z_scores >= z_threshold).astype(int)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(true_labels, predictions).ravel()
    
    # Calculate metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # ROC curve
    fpr, tpr, thresholds = roc_curve(true_labels, all_z_scores)
    roc_auc = auc(fpr, tpr)
    
    return {
        'confusion_matrix': {'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn},
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'detection_rate': tp / len(generated_z_scores),  # TPR for generated code
        'false_positive_rate': fp / len(original_z_scores)  # FPR for original code
    }

calculate_confusion_metrics(results_df['original_z_score'], results_df['generated_z_score'], Z_THRESHOLD)

In [ ]:
if results_df is not None:
    print("=== WATERMARK DETECTION PERFORMANCE ===\n")
    
    # CORRECTED APPROACH: Following recommended confusion matrix calculation
    
    # Extract z-scores
    original_z_scores = results_df['original_z_score'].fillna(0).tolist()
    generated_z_scores = results_df['generated_z_score'].fillna(0).tolist()
    
    # Ground Truth: Original code should be non-watermarked (0), Generated should be watermarked (1)
    true_labels = [0] * len(original_z_scores) + [1] * len(generated_z_scores)
    
    # Predictions: Based on z-score threshold
    original_predictions = [1 if z >= Z_THRESHOLD else 0 for z in original_z_scores]
    generated_predictions = [1 if z >= Z_THRESHOLD else 0 for z in generated_z_scores]
    all_predictions = original_predictions + generated_predictions
    
    # Z-scores for ROC calculation
    all_z_scores = original_z_scores + generated_z_scores
    
    # Convert to numpy arrays for easier calculation
    y_true = np.array(true_labels)
    y_pred = np.array(all_predictions)
    
    # Confusion matrix components
    tp = ((y_true == 1) & (y_pred == 1)).sum()   # Generated correctly detected as watermarked
    fp = ((y_true == 0) & (y_pred == 1)).sum()   # Original incorrectly detected as watermarked
    tn = ((y_true == 0) & (y_pred == 0)).sum()   # Original correctly detected as non-watermarked
    fn = ((y_true == 1) & (y_pred == 0)).sum()   # Generated incorrectly detected as non-watermarked
    
    print("Confusion Matrix Components:")
    print(f"TP (Generated correctly detected as watermarked): {tp}")
    print(f"FP (Original incorrectly detected as watermarked): {fp}")
    print(f"TN (Original correctly detected as non-watermarked): {tn}")
    print(f"FN (Generated incorrectly detected as non-watermarked): {fn}")
    print(f"Total samples: {tp + fp + tn + fn}")
    
    # Calculate metrics
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # Sensitivity/Recall
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0  # Specificity
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    f1 = 2 * (precision * tpr) / (precision + tpr) if (precision + tpr) > 0 else 0
    
    print(f"\nConfusion Matrix:")
    print(f"                    Predicted")
    print(f"                 WM    Not-WM")
    print(f"Actual    WM   {tp:4d}    {fn:4d}")
    print(f"       Not-WM  {fp:4d}    {tn:4d}")
    
    print(f"\nGround Truth Definition:")
    print(f"   Original code samples: Non-watermarked (0)")
    print(f"   Generated code samples: Watermarked (1)")
    print(f"   Total original samples: {len(original_z_scores)}")
    print(f"   Total generated samples: {len(generated_z_scores)}")
    
    print(f"\nPerformance Metrics:")
    print(f"TPR (Sensitivity): {tpr:.3f}  - How well we detect watermarks in generated code")
    print(f"FPR (Fall-out):    {fpr:.3f}  - How often we falsely detect watermarks in original code")
    print(f"TNR (Specificity): {tnr:.3f}  - How well we detect non-watermarks in original code")
    print(f"FNR (Miss rate):   {fnr:.3f}  - How often we miss watermarks in generated code")
    print(f"Precision:         {precision:.3f}  - Accuracy of watermark predictions")
    print(f"Accuracy:          {accuracy:.3f}  - Overall correctness")
    print(f"F1-Score:          {f1:.3f}  - Harmonic mean of precision & recall")
else:
    print("❌ No results data available for analysis")

### AUROC Calculation

In [ ]:
if results_df is not None and len(all_z_scores) > 0:
    # AUROC calculation using the corrected approach
    try:
        auc = roc_auc_score(y_true, all_z_scores)
        print(f"AUROC: {auc:.3f}  - Area under ROC curve")
        
        # Calculate ROC curve
        fpr_curve, tpr_curve, thresholds = roc_curve(y_true, all_z_scores)
        
        print(f"\nAUROC Interpretation:")
        if auc > 0.9:
            print(f"   Excellent discrimination (AUC > 0.9)")
        elif auc > 0.8:
            print(f"   Good discrimination (0.8 < AUC ≤ 0.9)")
        elif auc > 0.7:
            print(f"   Fair discrimination (0.7 < AUC ≤ 0.8)")
        elif auc > 0.6:
            print(f"   Poor discrimination (0.6 < AUC ≤ 0.7)")
        else:
            print(f"   Very poor discrimination (AUC ≤ 0.6)")
        
    except Exception as e:
        print(f"AUROC calculation failed: {e}")
        auc = None
        fpr_curve, tpr_curve, thresholds = None, None, None
else:
    auc = None
    fpr_curve, tpr_curve, thresholds = None, None, None
    print("AUROC: Not available (missing data)")

## 📊 Visualization

In [ ]:
if results_df is not None and auc is not None:
    plt.figure(figsize=(18, 6))
    
    # Plot 1: ROC Curve
    plt.subplot(1, 3, 1)
    plt.plot(fpr_curve, tpr_curve, linewidth=2, label=f'AUROC = {auc:.3f}')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
    plt.scatter([fpr], [tpr], color='red', s=100, zorder=5, 
                label=f'Current Threshold\n(FPR={fpr:.3f}, TPR={tpr:.3f})')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - Watermark Detection')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Plot 2: Z-score Distribution by Code Type
    plt.subplot(1, 3, 2)
    
    # Use corrected data: original (non-watermarked) vs generated (watermarked)
    plt.hist(original_z_scores, bins=15, alpha=0.7, 
             label=f'Original (Non-WM, n={len(original_z_scores)})', 
             color='red', density=True)
    plt.hist(generated_z_scores, bins=15, alpha=0.7, 
             label=f'Generated (WM, n={len(generated_z_scores)})', 
             color='blue', density=True)
    plt.axvline(x=Z_THRESHOLD, color='black', linestyle='--', linewidth=2,
                label=f'Threshold = {Z_THRESHOLD}')
    plt.xlabel('Z-Score')
    plt.ylabel('Density')
    plt.title('Z-Score Distribution by Code Type')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Plot 3: Threshold Analysis
    plt.subplot(1, 3, 3)
    thresholds_range = np.linspace(min(all_z_scores), max(all_z_scores), 50)
    
    tpr_values = []
    fpr_values = []
    
    for thresh in thresholds_range:
        # Recalculate predictions for this threshold
        thresh_predictions = np.array([1 if z >= thresh else 0 for z in all_z_scores])
        
        tp_thresh = ((y_true == 1) & (thresh_predictions == 1)).sum()
        fp_thresh = ((y_true == 0) & (thresh_predictions == 1)).sum()
        tn_thresh = ((y_true == 0) & (thresh_predictions == 0)).sum()
        fn_thresh = ((y_true == 1) & (thresh_predictions == 0)).sum()
        
        tpr_thresh = tp_thresh / (tp_thresh + fn_thresh) if (tp_thresh + fn_thresh) > 0 else 0
        fpr_thresh = fp_thresh / (fp_thresh + tn_thresh) if (fp_thresh + tn_thresh) > 0 else 0
        
        tpr_values.append(tpr_thresh)
        fpr_values.append(fpr_thresh)
    
    plt.plot(thresholds_range, tpr_values, label='TPR (Sensitivity)', linewidth=2, color='blue')
    plt.plot(thresholds_range, fpr_values, label='FPR (Fall-out)', linewidth=2, color='red')
    plt.axvline(x=Z_THRESHOLD, color='black', linestyle='--', alpha=0.7,
                label=f'Current Threshold = {Z_THRESHOLD}')
    plt.xlabel('Z-Score Threshold')
    plt.ylabel('Rate')
    plt.title('TPR vs FPR by Threshold')
    plt.legend()
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Find optimal threshold (Youden's J statistic: TPR - FPR)
    j_scores = np.array(tpr_values) - np.array(fpr_values)
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds_range[optimal_idx]
    optimal_tpr = tpr_values[optimal_idx]
    optimal_fpr = fpr_values[optimal_idx]
    
    print(f"\n=== THRESHOLD ANALYSIS ===")
    print(f"Current threshold: {Z_THRESHOLD}")
    print(f"Z-score range: [{min(all_z_scores):.3f}, {max(all_z_scores):.3f}]")
    print(f"Optimal threshold (max J = TPR - FPR): {optimal_threshold:.3f}")
    print(f"  At optimal threshold: TPR={optimal_tpr:.3f}, FPR={optimal_fpr:.3f}, J={j_scores[optimal_idx]:.3f}")
    print(f"  At current threshold: TPR={tpr:.3f}, FPR={fpr:.3f}, J={tpr-fpr:.3f}")
    
    if abs(optimal_threshold - Z_THRESHOLD) > 0.5:
        if optimal_threshold > Z_THRESHOLD:
            print(f"⚠️  Consider increasing threshold to {optimal_threshold:.2f} for better performance")
        else:
            print(f"⚠️  Consider decreasing threshold to {optimal_threshold:.2f} for better performance")
    else:
        print(f"✅ Current threshold is close to optimal")
    
else:
    print("📊 Visualization skipped - missing data or AUC calculation failed")

## 📝 Code Quality Analysis

In [ ]:
if results_df is not None and 'pass_rate' in results_df.columns:
    print("=== CODE QUALITY ANALYSIS ===\n")
    
    # Overall quality metrics
    avg_pass_rate = results_df['pass_rate'].mean()
    total_samples = len(results_df)
    all_passed = results_df['all_passed'].sum()
    
    print(f"📊 Overall Code Quality:")
    print(f"   Total samples: {total_samples}")
    print(f"   Average pass rate: {avg_pass_rate:.1f}%")
    print(f"   Samples with all tests passed: {all_passed} ({all_passed/total_samples:.1%})")
    
    # Quality comparison: Generated (watermarked) vs Original (non-watermarked) based on actual implementation
    # Note: In our case, we're analyzing generated code quality, so we compare based on watermark detection results
    
    # Use the corrected ground truth approach
    detected_as_watermarked = results_df['generated_z_score'] >= Z_THRESHOLD
    detected_as_non_watermarked = ~detected_as_watermarked
    
    wm_samples = results_df[detected_as_watermarked]
    no_wm_samples = results_df[detected_as_non_watermarked]
    
    if len(wm_samples) > 0 and len(no_wm_samples) > 0:
        wm_pass_rate = wm_samples['pass_rate'].mean()
        no_wm_pass_rate = no_wm_samples['pass_rate'].mean()
        
        print(f"\n🔍 Quality by Detection Status:")
        print(f"   Detected as watermarked ({len(wm_samples)} samples): {wm_pass_rate:.1f}% pass rate")
        print(f"   Detected as non-watermarked ({len(no_wm_samples)} samples): {no_wm_pass_rate:.1f}% pass rate")
        print(f"   Quality difference: {wm_pass_rate - no_wm_pass_rate:+.1f}% (WM - Non-WM)")
        
        # Statistical significance test (if samples are large enough)
        if len(wm_samples) >= 10 and len(no_wm_samples) >= 10:
            try:
                from scipy.stats import ttest_ind
                stat, p_value = ttest_ind(wm_samples['pass_rate'], no_wm_samples['pass_rate'])
                significance = "significant" if p_value < 0.05 else "not significant"
                print(f"   T-test p-value: {p_value:.4f} ({significance})")
                
                if p_value < 0.05:
                    if wm_pass_rate > no_wm_pass_rate:
                        print(f"   📈 Watermarked code performs significantly better")
                    else:
                        print(f"   📉 Watermarked code performs significantly worse")
                else:
                    print(f"   ➡️ No significant difference in code quality")
                    
            except ImportError:
                print(f"   T-test: scipy not available for statistical testing")
            except Exception as e:
                print(f"   T-test: Could not compute ({e})")
    
    # Quality distribution visualization
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.hist(results_df['pass_rate'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    plt.axvline(avg_pass_rate, color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {avg_pass_rate:.1f}%')
    plt.xlabel('Pass Rate (%)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Test Pass Rates')
    plt.legend()
    plt.grid(alpha=0.3)
    
    if len(wm_samples) > 0 and len(no_wm_samples) > 0:
        plt.subplot(1, 3, 2)
        plt.hist(no_wm_samples['pass_rate'], bins=15, alpha=0.7, 
                 label=f'Non-WM Detected (μ={no_wm_pass_rate:.1f}%)', color='red')
        plt.hist(wm_samples['pass_rate'], bins=15, alpha=0.7, 
                 label=f'WM Detected (μ={wm_pass_rate:.1f}%)', color='blue')
        plt.xlabel('Pass Rate (%)')
        plt.ylabel('Frequency')
        plt.title('Pass Rate by Detection Status')
        plt.legend()
        plt.grid(alpha=0.3)
        
        # Subplot 3: Quality vs Z-score
        plt.subplot(1, 3, 3)
        plt.scatter(results_df['generated_z_score'], results_df['pass_rate'], 
                   alpha=0.6, color='purple')
        plt.axvline(x=Z_THRESHOLD, color='red', linestyle='--', 
                   label=f'Threshold = {Z_THRESHOLD}')
        plt.xlabel('Z-Score')
        plt.ylabel('Pass Rate (%)')
        plt.title('Code Quality vs Z-Score')
        plt.legend()
        plt.grid(alpha=0.3)
        
        # Calculate correlation
        correlation = results_df['generated_z_score'].corr(results_df['pass_rate'])
        plt.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
                transform=plt.gca().transAxes, bbox=dict(boxstyle="round", facecolor='white'))
    
    plt.tight_layout()
    plt.show()
    
    # Additional quality insights
    print(f"\n📋 Quality Insights:")
    if 'generated_z_score' in results_df.columns:
        correlation = results_df['generated_z_score'].corr(results_df['pass_rate'])
        print(f"   Correlation between z-score and pass rate: {correlation:.3f}")
        
        if abs(correlation) > 0.3:
            direction = "positive" if correlation > 0 else "negative"
            strength = "strong" if abs(correlation) > 0.7 else "moderate"
            print(f"   📊 {strength.title()} {direction} correlation detected")
        else:
            print(f"   📊 Weak correlation between watermarking and quality")
    
else:
    print("📝 Code quality analysis skipped - missing test results")

## 🎯 Final Summary and Assessment

In [ ]:
if results_df is not None:
    print("=== WATERMARK DETECTION SUMMARY ===\n")
    
    print(f"📊 Dataset Overview:")
    print(f"   Total samples analyzed: {len(results_df)}")
    print(f"   Original (non-watermarked) samples: {len(original_z_scores)}")
    print(f"   Generated (watermarked) samples: {len(generated_z_scores)}")
    print(f"   Expected watermark rate: {len(generated_z_scores)/(len(original_z_scores)+len(generated_z_scores)):.1%}")
    
    print(f"\n🎯 Detection Performance:")
    if auc is not None:
        auc_assessment = ("Excellent" if auc > 0.9 else 
                         "Good" if auc > 0.8 else 
                         "Fair" if auc > 0.7 else "Poor")
        print(f"   AUROC: {auc:.3f} ({auc_assessment})")
    
    print(f"   Accuracy: {accuracy:.3f} ({accuracy:.1%})")
    print(f"   TPR (Sensitivity): {tpr:.3f} - Detected {tpr:.1%} of generated code as watermarked")
    print(f"   FPR (Fall-out): {fpr:.3f} - {fpr:.1%} of original code falsely flagged as watermarked")
    print(f"   TNR (Specificity): {tnr:.3f} - Correctly identified {tnr:.1%} of original code as non-watermarked")
    print(f"   Precision: {precision:.3f} - {precision:.1%} of watermark detections were correct")
    print(f"   F1-Score: {f1:.3f}")
    
    # Detection effectiveness interpretation
    print(f"\n🔍 Detection Effectiveness:")
    if tpr >= 0.9:
        print(f"   ✅ Excellent watermark detection rate ({tpr:.1%})")
    elif tpr >= 0.8:
        print(f"   ✅ Good watermark detection rate ({tpr:.1%})")
    elif tpr >= 0.7:
        print(f"   ⚠️ Fair watermark detection rate ({tpr:.1%})")
    else:
        print(f"   ❌ Poor watermark detection rate ({tpr:.1%})")
        
    if fpr <= 0.1:
        print(f"   ✅ Excellent false positive control ({fpr:.1%})")
    elif fpr <= 0.2:
        print(f"   ✅ Good false positive control ({fpr:.1%})")
    elif fpr <= 0.3:
        print(f"   ⚠️ Fair false positive control ({fpr:.1%})")
    else:
        print(f"   ❌ Poor false positive control ({fpr:.1%})")
    
    if 'pass_rate' in results_df.columns:
        print(f"\n📝 Code Quality Impact:")
        print(f"   Overall pass rate: {avg_pass_rate:.1f}%")
        if len(wm_samples) > 0 and len(no_wm_samples) > 0:
            print(f"   Watermark detected code: {wm_pass_rate:.1f}% pass rate")
            print(f"   Non-watermark detected code: {no_wm_pass_rate:.1f}% pass rate")
            quality_impact = wm_pass_rate - no_wm_pass_rate
            print(f"   Quality impact: {quality_impact:+.1f}%")
            
            if abs(quality_impact) < 5:
                print(f"   📊 Minimal quality impact from watermarking")
            elif quality_impact > 0:
                print(f"   📈 Watermarked code shows better quality")
            else:
                print(f"   📉 Watermarked code shows reduced quality")
    
    # Overall assessment
    if auc is not None:
        if auc >= 0.9 and fpr <= 0.1 and tpr >= 0.8:
            assessment = "🟢 EXCELLENT: High detection accuracy with low false positives"
        elif auc >= 0.8 and fpr <= 0.2 and tpr >= 0.7:
            assessment = "🟡 GOOD: Reasonable detection with acceptable false positive rate"
        elif auc >= 0.7 and tpr >= 0.6:
            assessment = "🟠 FAIR: Moderate detection capability, needs improvement"
        else:
            assessment = "🔴 POOR: Low detection capability, significant improvement needed"
    else:
        assessment = "❓ ASSESSMENT: Incomplete data for full evaluation"
    
    print(f"\n📈 Overall Assessment: {assessment}")
    
    print(f"\n🔧 Recommendations:")
    if fpr > 0.2:
        print(f"   • Increase threshold to reduce false positives (current: {Z_THRESHOLD})")
        print(f"   • Consider stronger watermark signal to improve discrimination")
    if tpr < 0.8:
        print(f"   • Strengthen watermarking technique to improve detection rate")
        print(f"   • Consider decreasing threshold if false positive rate allows")
    if auc is not None and auc < 0.8:
        print(f"   • Review watermarking strategy - poor separation between classes")
        print(f"   • Consider different watermarking approach or parameters")
    if 'pass_rate' in results_df.columns and avg_pass_rate < 80:
        print(f"   • Investigate code quality issues - average pass rate is low ({avg_pass_rate:.1f}%)")
    
    # Threshold recommendation
    if 'optimal_threshold' in locals() and abs(optimal_threshold - Z_THRESHOLD) > 0.5:
        print(f"   • Consider adjusting threshold from {Z_THRESHOLD} to {optimal_threshold:.2f}")
    
    print(f"\n✅ Analysis completed successfully!")
    print(f"   Results saved to: {RESULTS_CSV}")
    print(f"\n📋 Key Findings:")
    print(f"   • Watermark detection works with {accuracy:.1%} accuracy")
    print(f"   • {tpr:.1%} of watermarked code correctly identified")
    print(f"   • {fpr:.1%} false positive rate on original code")
    if auc is not None:
        print(f"   • AUROC of {auc:.3f} indicates {auc_assessment.lower()} discriminative ability")
    
else:
    print("❌ No results available for final summary")

## 💾 Save Results

Export results for further analysis or reporting:

In [ ]:
if results_df is not None:
    # Create summary metrics dictionary
    summary_metrics = {
        'dataset_size': len(results_df),
        'watermarked_samples': int(y_true.sum()),
        'z_threshold': Z_THRESHOLD,
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall_tpr': float(tpr),
        'specificity_tnr': float(tnr),
        'fpr': float(fpr),
        'fnr': float(fnr),
        'f1_score': float(f1),
        'auroc': float(auc) if auc is not None else None
    }
    
    if 'pass_rate' in results_df.columns:
        summary_metrics.update({
            'avg_pass_rate': float(avg_pass_rate),
            'wm_pass_rate': float(wm_pass_rate) if len(wm_samples) > 0 else None,
            'no_wm_pass_rate': float(no_wm_pass_rate) if len(no_wm_samples) > 0 else None
        })
    
    # Save summary metrics
    summary_path = RESULTS_CSV.replace('.csv', '_summary.json')
    import json
    with open(summary_path, 'w') as f:
        json.dump(summary_metrics, f, indent=2)
    
    print(f"📊 Summary metrics saved to: {summary_path}")
    
    # Show available outputs
    print(f"\n📁 Output Files:")
    print(f"   📄 Detailed results: {RESULTS_CSV}")
    print(f"   📄 Summary metrics: {summary_path}")
    print(f"   📁 Generated code: {OUTPUT_DIR}")
    
    # Display final metrics table
    print(f"\n📋 Final Metrics Table:")
    metrics_df = pd.DataFrame([
        ['True Positives (TP)', tp, 'Watermarks correctly detected'],
        ['False Positives (FP)', fp, 'Non-watermarks incorrectly flagged'],
        ['True Negatives (TN)', tn, 'Non-watermarks correctly identified'],
        ['False Negatives (FN)', fn, 'Watermarks missed'],
        ['Accuracy', f'{accuracy:.3f}', 'Overall correctness'],
        ['Precision', f'{precision:.3f}', 'Accuracy of watermark predictions'],
        ['Recall (TPR)', f'{tpr:.3f}', 'Watermark detection rate'],
        ['Specificity (TNR)', f'{tnr:.3f}', 'Non-watermark detection rate'],
        ['F1-Score', f'{f1:.3f}', 'Harmonic mean of precision & recall'],
        ['AUROC', f'{auc:.3f}' if auc else 'N/A', 'Area under ROC curve']
    ], columns=['Metric', 'Value', 'Description'])
    
    print(metrics_df.to_string(index=False))

else:
    print("💾 No results to save")